In [17]:
import os
import zstandard  # pip install zstandard
from tqdm import tqdm

In [18]:
def zst_files_in_dir(directory):
    """List all .zst files in a directory."""
    files = []
    for filename in os.listdir(directory):
        if filename.endswith(".zst") and os.path.isfile(os.path.join(directory, filename)):
            files.append(filename)
    return files

In [ ]:
def decompress_zst_to_text(input_file, output_file, vocab):
    """Decompresses a .zst file and extracts text to output_file, updating vocab."""
    with open(input_file, "rb") as infile:
        dctx = zstandard.ZstdDecompressor()
        with dctx.stream_reader(infile) as reader:
            current_line = b""
            while True:
                chunk = reader.read(16384)  # Read in 16KB chunks
                if not chunk:
                    break
                current_line += chunk
                # Split into lines (handles partial lines)
                lines = current_line.split(b"\n")
                current_line = lines.pop()
                # Process each line
                for line in lines:
                    try:
                        text = line.decode("utf-8").strip()
                        if text:
                            yield text
                            characters = set(text)
                            vocab.update(characters)
                    except UnicodeDecodeError:
                        pass  # Skip invalid UTF-8 sequences
            # Process the remaining partial line
            if current_line:
                try:
                    text = current_line.decode("utf-8").strip()
                    if text:
                        yield text
                        characters = set(text)
                        vocab.update(characters)
                except UnicodeDecodeError:
                    pass

In [19]:
folder_path = "openwebtext2"
output_file_train = "output_train_v3.txt"
output_file_val = "output_val_v3.txt"
vocab_file = "vocab_v3.txt"


In [20]:
# Gather files
files = zst_files_in_dir(folder_path)
total_files = len(files)
print(f"Total files: {total_files}")

20610


In [21]:
# Split files into train/val (90%/10%)
split_index = int(total_files * 0.9)
files_train = files[:split_index]
files_val = files[split_index:]
vocab = set()

In [ ]:
# Process training files
with open(output_file_train, "w", encoding="utf-8") as outf:
    for filename in tqdm(files_train, total=len(files_train), desc="Processing Train"):
        file_path = os.path.join(folder_path, filename)
        try:
            for text in decompress_zst_to_text(file_path, outf.name, vocab):
                outf.write(text + "\n")
        except Exception as e:
            print(f"Error processing {file_path}: {e}")

In [23]:
# Process validation files
with open(output_file_val, "w", encoding="utf-8") as outf:
    for filename in tqdm(files_val, total=len(files_val), desc="Processing Val"):
        file_path = os.path.join(folder_path, filename)
        try:
            for text in decompress_zst_to_text(file_path, outf.name, vocab):
                outf.write(text + "\n")
        except Exception as e:
            print(f"Error processing {file_path}: {e}")


100%|█████████████████████████████████████████████████| 2061/2061 [03:01<00:00, 11.34it/s]


In [ ]:
# Write vocabulary
with open(vocab_file, "w", encoding="utf-8") as vfile:
    for char in sorted(vocab):
        vfile.write(char + "\n")
        